# 🚗 Automatic Number Plate Recognition (ANPR) — YOLOv8 Training
### Project: Computer Vision Based Road Safety Monitoring System
**Author:** Shibchandan Mistry  
**Module:** ANPR (License Plate Detection & Text Recognition)
**Dataset:** Kaggle `andrewmvd/car-plate-detection` (Downloaded via `kagglehub` — no API key needed)

---
> **Tip:** Before running, ensure you have enabled GPU in Colab:  
> Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.

## Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

## Step 2: Install Ultralytics (YOLOv8) & Kagglehub

In [ ]:
!pip install -q ultralytics kagglehub

## Step 3: Download Kaggle Dataset & Auto-Convert to YOLOv8 Format
Downloads `andrewmvd/car-plate-detection` via `kagglehub` (no login/API key required) and converts Pascal VOC XML annotations to YOLOv8 normalized coordinates.

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
import kagglehub
from sklearn.model_selection import train_test_split

print("📥 Downloading dataset from Kaggle via kagglehub...")
path = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print(f"Downloaded raw files to: {path}")

# Setup target YOLO directories
yolo_dir = Path("/content/dataset_yolo")
for split in ["train", "val"]:
    (yolo_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

images_dir = Path(path) / "images"
annotations_dir = Path(path) / "annotations"

xml_files = sorted(list(annotations_dir.glob("*.xml")))
train_xmls, val_xmls = train_test_split(xml_files, test_size=0.2, random_state=42)

def convert_to_yolo(xml_list, split_name):
    for xml_file in xml_list:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        size = root.find("size")
        width = float(size.find("width").text)
        height = float(size.find("height").text)
        
        img_name = root.find("filename").text
        src_img = images_dir / img_name
        
        if not src_img.exists():
            continue
            
        # Copy image
        shutil.copy(src_img, yolo_dir / "images" / split_name / img_name)
        
        # Convert bounding boxes
        label_file = yolo_dir / "labels" / split_name / f"{src_img.stem}.txt"
        with open(label_file, "w") as f:
            for obj in root.findall("object"):
                bndbox = obj.find("bndbox")
                xmin = float(bndbox.find("xmin").text)
                ymin = float(bndbox.find("ymin").text)
                xmax = float(bndbox.find("xmax").text)
                ymax = float(bndbox.find("ymax").text)
                
                x_center = ((xmin + xmax) / 2.0) / width
                y_center = ((ymin + ymax) / 2.0) / height
                w = (xmax - xmin) / width
                h = (ymax - ymin) / height
                
                f.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

print("🔄 Converting Pascal VOC annotations to YOLOv8 format...")
convert_to_yolo(train_xmls, "train")
convert_to_yolo(val_xmls, "val")

# Create data.yaml
yaml_content = f"""path: {yolo_dir}
train: images/train
val: images/val

names:
  0: license_plate
"""

data_yaml = str(yolo_dir / "data.yaml")
with open(data_yaml, "w") as f:
    f.write(yaml_content)

print(f"\n✅ Success! Dataset ready for YOLOv8 at: {data_yaml}")

## Step 4: Train YOLOv8 Nano Model
- **Base model:** `yolov8n.pt`
- **Epochs:** `30` (takes ~4–6 minutes on free T4 GPU)
- **Image size:** `640`
- **Batch size:** `16`

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Start Training
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    project="anpr_training",
    name="yolov8n_plate",
    device=0
)

print("\n✅ Training complete!")

## Step 5: View Training Metrics & Accuracy Graphs
Save these graphs for your **Project Report & PPT Presentation**.

In [ ]:
from IPython.display import Image, display
import os

results_img = "anpr_training/yolov8n_plate/results.png"
conf_matrix = "anpr_training/yolov8n_plate/confusion_matrix.png"

if os.path.exists(results_img):
    print("--- Training & Validation Metrics (Loss, Precision, Recall, mAP) ---")
    display(Image(results_img, width=800))

if os.path.exists(conf_matrix):
    print("--- Confusion Matrix ---")
    display(Image(conf_matrix, width=600))

## Step 6: Test Detection on Validation Set

In [ ]:
trained_model = YOLO("anpr_training/yolov8n_plate/weights/best.pt")

metrics = trained_model.val(data=data_yaml)

print(f"\nEvaluation Results:")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print(f"  mAP@0.50      : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95 : {metrics.box.map:.4f}")

## Step 7: Download `best.pt` to Use in Your Local VS Code Project
Downloads your trained weights to your browser. Put this file at: `results/anpr/weights/best.pt`.

In [ ]:
from google.colab import files

weights_path = "anpr_training/yolov8n_plate/weights/best.pt"
print(f"Downloading {weights_path} to your computer...")
files.download(weights_path)